In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from walinet.parameter_calibration.load_data import *

from walinet.parameter_calibration.metab_normalization import *

from walinet.parameter_calibration.compute_statistics import (
    extract_valid_voxels,
    calculate_pooled_median_iqr,
)

from walinet.parameter_calibration.metab_calibration import *

from walinet.parameter_calibration.water_lipid_ratios import *

from walinet.parameter_calibration.pipeline_FWHM_SNR_shifts import (
    calibrate_parameter_from_maps,
)

In [ ]:
bandwidth_hz = 939.85
nmr_frequency_hz = 123231706.0
water_ppm = 4.68

In [ ]:
TRAIN_CONFIG_PATH = "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/configs/Training/train_3T.yaml"


SUBJECT_DIRS = [
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/ClimaX_Brisbane/Vol4/Res36x36/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/ClimaX_Brisbane/Vol4/Res50x50/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/PRISMAFIT_Vienna/Vol01_BS/Res36x36/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/PRISMAFIT_Vienna/Vol01_BS/Res50x50/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/PRISMAFIT_Vienna/Vol01_BS/Res64x64x41/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/PRISMA_Vienna/Vol01_WB/Res36x36/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/PRISMA_Vienna/Vol01_WB/Res50x50/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/PRISMA_Vienna/Vol01_WB/Res64x64x41/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/VIDA_Vienna/Vol01_PW/Res36x36/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/VIDA_Vienna/Vol01_PW/Res50x50/OriginalData",
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/data/3T/VIDA_Vienna/Vol01_PW/Res64x64x41/OriginalData"    
]

In [ ]:
calibration_maps = load_calibration_maps(
    SUBJECT_DIRS,
    suffix=".nii.gz",  # alternativ: ".mnc"
)

for subject_id, subject_data in calibration_maps.items():
    print(
        subject_id,
        len(subject_data["metabolites"]),
        "Metaboliten, FWHM:",
        subject_data["fwhm"].shape,
    )

In [ ]:
r_calibration = calculate_r_maps(
    calibration_maps=calibration_maps,
    train_config_path=TRAIN_CONFIG_PATH,
)

In [ ]:
# #optionally plot NAA vs FWHM to make sure they oriented consistently!

# fwhm_naa_figures = plot_fwhm_vs_metabolite_slices(
#     calibration_maps,
#     metabolite_name="NAA",
#     slices_per_figure=6,
#     slice_axis=2,
#     percentile_range=(1, 99),
#   # save_dir="SavedGraphics/FWHM_vs_NAA",
#     show=True,
# )

In [ ]:
Metabos = r_calibration[
    next(iter(r_calibration))
]["matched_basis_names"]

for METABO in Metabos:
    calibration = calibrate_metabolite_ratio_from_r_maps(
        r_calibration=r_calibration,
        metabolite_name=METABO,
        bins=50,
        plot_percentile=99.5,
        truncated_normal_sigma_factor=2.0,
        lognormal_sigma_factor=2.0,
        save_path=f"SavedGraphics/{METABO}_ratio_calibration.pdf",
        show=True,
    )